In [1]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_excel(r"")

In [ ]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

In [ ]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [ ]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [ ]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready